In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# save_path = "/content/drive/MyDrive/aclImdb"
# zip_path  = "/content/drive/MyDrive/aclImdb_v1.tar.gz"
# import os
# if not os.path.exists(save_path):
#     os.makedirs(save_path)
# !wget https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz -O $zip_path
# !tar -xzf $zip_path -C /content/drive/MyDrive/

# print(" 다운로드 완료! Google Drive에 저장됨.")

#  완료되었는지 체크
# !ls /content/drive/MyDrive/aclImdb/train/pos | head

In [ ]:
import os
import tarfile
import requests

# 1. 데이터 저장할 루트 경로 (네 컴퓨터 경로)
DATA_ROOT = r"/content/drive/MyDrive/Colab Notebooks"

# 2. 작업 폴더를 이 위치로 이동
os.chdir(DATA_ROOT)

# 3. 다운로드할 URL과 저장할 파일/폴더 경로 설정
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
zip_path = os.path.join(DATA_ROOT, "aclImdb_v1.tar.gz")
target_dir = os.path.join(DATA_ROOT, "aclImdb")

# 4. 이미 aclImdb 폴더가 없으면 다운로드 + 압축 풀기
if not os.path.exists(target_dir):
    # (1) tar.gz 파일이 없으면 다운로드
    if not os.path.exists(zip_path):
        print("📥 IMDB 데이터 다운로드 중...")
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(zip_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        print("✅ 다운로드 완료:", zip_path)

    # (2) 압축 풀기
    print("📦 압축 해제 중...")
    with tarfile.open(zip_path, "r:gz") as tar:
        tar.extractall(path=DATA_ROOT)
    print("✅ 압축 해제 완료! aclImdb 폴더 생성됨")

else:
    print("✅ 이미 aclImdb 폴더가 존재합니다:", target_dir)

# 5. 제대로 풀렸는지 간단 체크 (앞 몇 개 파일만 보기)
pos_dir = os.path.join(target_dir, "train", "pos")
print("🔍 예시 파일들:", os.listdir(pos_dir)[:5])

✅ 이미 aclImdb 폴더가 존재합니다: /content/drive/MyDrive/Colab Notebooks/aclImdb
🔍 예시 파일들: ['11543_8.txt', '11561_10.txt', '11761_10.txt', '11371_10.txt', '11472_7.txt']


In [ ]:
!pip install torchtext==0.17.2 --force-reinstall
!pip install transformers==4.38.2 --quite
!pip install accelerate==0.27.2 --quite

  Using cached torchtext-0.17.2-cp312-cp312-manylinux1_x86_64.whl.metadata (7.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (25 kB)
  Using cached numpy-2.3.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvi


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: --quite


KeyboardInterrupt: 

In [ ]:
import torchtext
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.nn as nn
from tqdm import tqdm
import math
import os, shutil, torch, numpy as np, pandas as pd


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

In [ ]:
# DATA_ROOT = r"/content/drive/MyDrive/Colab Notebooks"
# os.chdir(DATA_ROOT)
# seed=1234
# np.random.seed(seed)
# torch.manual_seed(seed)
# torch.cuda.manual_seed(seed)
# torch.backends.cudnn.deterministic=True

In [ ]:
seed = 1234
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

IMDB_SRC = "/content/drive/MyDrive/Colab Notebooks/aclImdb"
IMDB_DST = "/content/aclImdb"

if not os.path.exists(IMDB_DST):
    import time
    print("📌 Copying IMDB dataset from Drive...")
    start = time.time()
    shutil.copytree(IMDB_SRC, IMDB_DST)
    print(f"✅ Copy done in {time.time()-start:.2f} seconds!")
else:
    print("✅ IMDB already copied:", IMDB_DST)

✅ IMDB already copied: /content/aclImdb


In [ ]:
DATA_ROOT = "/content"
os.chdir(DATA_ROOT)

def load_imdb_split(split_dir):
    texts, labels = [], []
    for label in ['pos', 'neg']:
        folder = os.path.join(split_dir, label)
        for filename in os.listdir(folder):
            with open(os.path.join(folder, filename), 'r', encoding='utf-8') as f:
                texts.append(f.read())
                labels.append(1 if label == 'pos' else 0)
    return pd.DataFrame({'text': texts, 'label': labels})

print("📂 Loading train/test...")
train_df = load_imdb_split(os.path.join(IMDB_DST, "train"))
print("  train_df loaded:", len(train_df))
test_df  = load_imdb_split(os.path.join(IMDB_DST, "test"))
print("  test_df loaded:", len(test_df))


📂 Loading train/test...
  train_df loaded: 25000
  test_df loaded: 25000


In [ ]:
tokenizer = get_tokenizer("basic_english")

def yield_tokens(text_series):
    for text in text_series:
        yield tokenizer(text)

vocab = build_vocab_from_iterator(
    yield_tokens(train_df['text']),
    specials=["<unk>"]
)
vocab.set_default_index(vocab["<unk>"])

MAX_LEN = 256
def encode_text(text):
    tokens = tokenizer(text)
    return vocab(tokens[:MAX_LEN])

print("🔠 Encoding texts...")
train_df['input_ids'] = train_df['text'].apply(encode_text)
test_df['input_ids']  = test_df['text'].apply(encode_text)
print("  encoding done")

def create_tensor_dataset(inputs, labels):
    tensor_seqs = [torch.tensor(x) for x in inputs]
    padded_seqs = pad_sequence(tensor_seqs, batch_first=True)
    labels_tensor = torch.tensor(list(labels), dtype=torch.long)
    return TensorDataset(padded_seqs, labels_tensor)

train_dataset = create_tensor_dataset(train_df['input_ids'], train_df['label'])
test_dataset  = create_tensor_dataset(test_df['input_ids'],  test_df['label'])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64)

print("✅ DataLoaders ready")


🔠 Encoding texts...
  encoding done
✅ DataLoaders ready


In [ ]:
for batch_x, batch_y in train_loader:
    print("Input batch shape:", batch_x.shape)
    print("Example token IDs:", batch_x[0][:20])
    print("Label:", batch_y[0].item())
    break


Input batch shape: torch.Size([64, 256])
Example token IDs: tensor([ 2064, 19872,    24,   959,  5808,    23,     9,  2920,    28,     7,
           33,  2569,    13,   955,     2,    28,    69,     9,  2064,     5])
Label: 0


In [ ]:
class SimpleLSTMClassifier(nn.Module):
  def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128, output_dim=1):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size, embedding_dim, padding_idx=vocab["<unk>"])
    self.lstm=nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, output_dim)
    self.sigmoid=nn.Sigmoid()
  def forward(self, x):
    embedded=self.embedding(x)
    output, (hidden, _) =self.lstm(embedded)
    hidden= hidden.squeeze(0)
    return self.sigmoid(self.fc(hidden))
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=SimpleLSTMClassifier(vocab_size=len(vocab), embedding_dim=300, hidden_dim=128).to(device)
criterion=nn.BCELoss()
optimizer=torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
from tqdm import tqdm

def train_loop(model, train_loader, test_loader, optimizer, criterion, device, epochs=5):
    model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        total_correct = 0

        print(f"\n Epoch {epoch}/{epochs}")

        progress_bar = tqdm(enumerate(train_loader),
                            total=len(train_loader),
                            desc="Training")

        for batch_idx, (x_batch, y_batch) in progress_bar:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device).float()

            optimizer.zero_grad()
            outputs = model(x_batch).squeeze(1)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            predictions = (outputs >= 0.5).long()
            total_correct += (predictions == y_batch).sum().item()

            progress_bar.set_postfix(loss=loss.item())

        train_acc = total_correct / len(train_loader.dataset)
        avg_loss = total_loss / len(train_loader)

        model.eval()
        with torch.no_grad():
            correct = 0
            for x_batch, y_batch in test_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device).float()
                outputs = model(x_batch).squeeze(1)
                preds = (outputs >= 0.5).long()
                correct += (preds == y_batch).sum().item()

        test_acc = correct / len(test_loader.dataset)

        print(f"Epoch {epoch} Summary - Avg Loss: {avg_loss:.4f}, "
              f"Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

    return model

In [ ]:
model = train_loop(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10,
)


 Epoch 1/10


Training: 100%|██████████| 391/391 [00:54<00:00,  7.19it/s, loss=0.703]


Epoch 1 Summary - Avg Loss: 0.6933, Train Acc: 0.5072, Test Acc: 0.5218

 Epoch 2/10


Training: 100%|██████████| 391/391 [00:57<00:00,  6.85it/s, loss=0.654]


Epoch 2 Summary - Avg Loss: 0.6737, Train Acc: 0.5608, Test Acc: 0.5330

 Epoch 3/10


Training: 100%|██████████| 391/391 [00:55<00:00,  7.09it/s, loss=0.428]


... (11 outputs left out)

Training: 100%|██████████| 391/391 [00:50<00:00,  7.67it/s, loss=0.246]


Epoch 9 Summary - Avg Loss: 0.3539, Train Acc: 0.8340, Test Acc: 0.7056

 Epoch 10/10


Training: 100%|██████████| 391/391 [00:50<00:00,  7.74it/s, loss=0.32]


Epoch 10 Summary - Avg Loss: 0.3606, Train Acc: 0.8365, Test Acc: 0.7207


In [ ]:
def predict_sentiment(text, model, vocab, tokenizer, device, threshold=0.5, max_len=256):
    model.eval()
    tokens = tokenizer(text)
    token_ids = vocab(tokens[:max_len])
    input_tensor = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor).item()
    label = "pos" if output >= threshold else "neg"
    return {"label": label, "probability": output}

In [ ]:
text = "This movie was a masterpiece. I loved every minute of it!"
result = predict_sentiment(text, model, vocab, tokenizer, device)
print(result)
text = "This film is great!"
result = predict_sentiment(text, model, vocab, tokenizer, device)
print(result)
text = "This film is terrible!"
result = predict_sentiment(text, model, vocab, tokenizer, device)
print(result)
text = "This film is not great, it's terrible!"
result = predict_sentiment(text, model, vocab, tokenizer, device)
print(result)

{'label': 'pos', 'probability': 0.893050491809845}
{'label': 'pos', 'probability': 0.9181540608406067}
{'label': 'neg', 'probability': 0.0017416616901755333}
{'label': 'neg', 'probability': 0.014283282682299614}


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleLSTMClassifier(vocab_size=len(vocab),embedding_dim=300,hidden_dim=128,).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
vectors = torchtext.vocab.GloVe(name="6B", dim=300, cache="~/.vector_cache")
pretrained_embedding = vectors.get_vecs_by_tokens(vocab.get_itos())
model.embedding.weight.data = pretrained_embedding

~/.vector_cache/glove.6B.zip: 862MB [02:39, 5.42MB/s]                           
100%|█████████▉| 399999/400000 [00:26<00:00, 14891.71it/s]


In [ ]:
model = train_loop(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10,
)


 Epoch 1/10


Training: 100%|██████████| 391/391 [00:55<00:00,  7.06it/s, loss=0.696]


Epoch 1 Summary - Avg Loss: 0.6878, Train Acc: 0.5274, Test Acc: 0.5288

 Epoch 2/10


Training: 100%|██████████| 391/391 [00:55<00:00,  7.01it/s, loss=0.578]


Epoch 2 Summary - Avg Loss: 0.6278, Train Acc: 0.6517, Test Acc: 0.6482

 Epoch 3/10


Training: 100%|██████████| 391/391 [00:51<00:00,  7.55it/s, loss=0.474]


... (11 outputs left out)

Training: 100%|██████████| 391/391 [00:51<00:00,  7.63it/s, loss=0.00471]


Epoch 9 Summary - Avg Loss: 0.0399, Train Acc: 0.9906, Test Acc: 0.8203

 Epoch 10/10


Training: 100%|██████████| 391/391 [00:51<00:00,  7.59it/s, loss=0.182]


Epoch 10 Summary - Avg Loss: 0.0247, Train Acc: 0.9946, Test Acc: 0.8123


In [ ]:
# class StrongLSTMClassifier(nn.Module):
#     def __init__(self,vocab_size, embedding_dim=300, hidden_dim=192, num_layers=2, bidirectional=True, dropout=0.5, embed_dropout=0.2, output_dim=1,):
#         super().__init__()
#         self.bidirectional = bidirectional
#         self.embedding = nn.Embedding(
#             vocab_size,
#             embedding_dim,
#             padding_idx=vocab["<unk>"],
#         )
#         self.embedding_dropout = nn.Dropout(embed_dropout)
#         self.lstm = nn.LSTM(
#             input_size=embedding_dim,
#             hidden_size=hidden_dim,
#             num_layers=num_layers,
#             batch_first=True,
#             bidirectional=bidirectional,
#             dropout=dropout if num_layers > 1 else 0.0,
#         )
#         factor = 2 if bidirectional else 1
#         fc_input_dim = hidden_dim * factor * 2
#         self.norm = nn.LayerNorm(fc_input_dim)
#         self.dropout = nn.Dropout(dropout)
#         self.fc = nn.Linear(fc_input_dim, output_dim)
#         self.sigmoid = nn.Sigmoid()
#     def forward(self, x):
#         embedded = self.embedding(x)
#         embedded = self.embedding_dropout(embedded)
#         output, (hidden, _) = self.lstm(embedded)
#         if self.bidirectional:
#             h_forward = hidden[-2]
#             h_backward = hidden[-1]
#             h_cat = torch.cat((h_forward, h_backward), dim=1)
#         else:
#             h_cat = hidden[-1]
#         pooled = torch.max(output, dim=1).values
#         h_final = torch.cat([h_cat, pooled], dim=1)
#         h_final = self.norm(h_final)
#         h_final = self.dropout(h_final)
#         logits = self.fc(h_final)
#         probs = self.sigmoid(logits)
#         return probs


In [ ]:
class StrongLSTMClassifier(nn.Module):
    def __init__(self,vocab_size, embedding_dim=300, hidden_dim=192, num_layers=2, bidirectional=True, dropout=0.55, embed_dropout=0.25, output_dim=1,):
        super().__init__()
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=vocab["<unk>"],
        )
        self.embed_norm = nn.LayerNorm(embedding_dim)
        self.embedding_dropout = nn.Dropout(embed_dropout)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        factor = 2 if bidirectional else 1
        fc_dim = hidden_dim * factor * 2
        self.lstm_norm = nn.LayerNorm(hidden_dim * factor)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(fc_dim, output_dim)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.embed_norm(embedded)
        embedded = self.embedding_dropout(embedded)
        output, (hidden, _) = self.lstm(embedded)
        if self.bidirectional:
            h_forward = hidden[-2]
            h_backward = hidden[-1]
            h_cat = torch.cat((h_forward, h_backward), dim=1)
        else:
            h_cat = hidden[-1]
        pooled = torch.max(output, dim=1).values
        h_cat = self.lstm_norm(h_cat)
        h_final = torch.cat([h_cat, pooled], dim=1)
        h_final = self.dropout(h_final)
        logits = self.fc(h_final)
        probs = self.sigmoid(logits)
        return probs

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = StrongLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=300,
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
model = train_loop(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10,
)


 Epoch 1/10


Training: 100%|██████████| 391/391 [03:06<00:00,  2.10it/s, loss=0.495]


Epoch 1 Summary - Avg Loss: 0.6018, Train Acc: 0.6550, Test Acc: 0.8182

 Epoch 2/10


Training: 100%|██████████| 391/391 [03:08<00:00,  2.07it/s, loss=0.273]


Epoch 2 Summary - Avg Loss: 0.3554, Train Acc: 0.8467, Test Acc: 0.8689

 Epoch 3/10


Training: 100%|██████████| 391/391 [03:09<00:00,  2.07it/s, loss=0.0799]


... (11 outputs left out)

Training: 100%|██████████| 391/391 [03:13<00:00,  2.03it/s, loss=0.0857]


Epoch 9 Summary - Avg Loss: 0.0431, Train Acc: 0.9846, Test Acc: 0.8692

 Epoch 10/10


Training: 100%|██████████| 391/391 [03:12<00:00,  2.03it/s, loss=0.013]


Epoch 10 Summary - Avg Loss: 0.0344, Train Acc: 0.9878, Test Acc: 0.8705


In [ ]:
def train_loop(model, train_loader, test_loader, optimizer, criterion, device, epochs=5):
    model.to(device)
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0

        print(f"\nEpoch {epoch}/{epochs}")
        progress_bar = tqdm(enumerate(train_loader),
                            total=len(train_loader),
                            desc="Training")

        for batch_idx, (x_batch, y_batch) in progress_bar:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device).float()

            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()
            total_correct += (preds == y_batch.long()).sum().item()

            progress_bar.set_postfix(loss=loss.item())

        train_acc = total_correct / len(train_loader.dataset)
        avg_loss = total_loss / len(train_loader)

        model.eval()
        with torch.no_grad():
            correct = 0
            for x_batch, y_batch in test_loader:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device).float()

                logits = model(x_batch)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).long()
                correct += (preds == y_batch.long()).sum().item()

        test_acc = correct / len(test_loader.dataset)
        print(
            f"Epoch {epoch} Summary - Avg Loss: {avg_loss:.4f}, "
            f"Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}"
        )

    return model


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)


In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self,vocab_size,embedding_dim=300,num_heads=4,hidden_dim=256,num_layers=2,
        num_classes=1,max_len=256,dropout=0.2,pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(embedding_dim, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        pad_mask = (x == self.pad_idx)
        embedded = self.embedding(x)
        encoded  = self.pos_encoder(embedded)
        out = self.transformer_encoder(encoded, src_key_padding_mask=pad_mask)

        non_pad_mask = (~pad_mask).unsqueeze(-1)
        masked_out = out * non_pad_mask
        lengths = non_pad_mask.sum(dim=1).clamp(min=1)
        pooled = masked_out.sum(dim=1) / lengths

        logits = self.classifier(pooled).squeeze(1)
        return logits
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# pad_idx = vocab["<pad>"]
# small_x = []
# small_y = []

# for xb, yb in train_loader:
#     small_x.append(xb)
#     small_y.append(yb)
#     if len(torch.cat(small_x)) >= 512:
#         break

# small_x = torch.cat(small_x)[:512]
# small_y = torch.cat(small_y)[:512].float()

# small_dataset = torch.utils.data.TensorDataset(small_x, small_y)
# small_loader = torch.utils.data.DataLoader(
#     small_dataset, batch_size=32, shuffle=True
# )

# small_model = TransformerClassifier(
#     vocab_size=len(vocab),
#     embedding_dim=300,
#     num_heads=4,
#     hidden_dim=256,
#     num_layers=2,
#     pad_idx=pad_idx,
# ).to(device)
# criterion = nn.BCEWithLogitsLoss()
# optimizer = torch.optim.Adam(small_model.parameters(), lr=1e-3)
# for epoch in range(1, 11):
#     small_model.train()
#     total_loss = 0.0
#     total_correct = 0
#     for xb, yb in small_loader:
#         xb = xb.to(device)
#         yb = yb.to(device).float()
#         optimizer.zero_grad()
#         logits = small_model(xb)
#         loss = criterion(logits, yb)
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#         probs = torch.sigmoid(logits)
#         preds = (probs >= 0.5).long()
#         total_correct += (preds == yb.long()).sum().item()
#     avg_loss = total_loss / len(small_loader)
#     acc = total_correct / len(small_dataset)
#     print(f"Epoch {epoch}: loss={avg_loss:.4f}, acc={acc:.4f}")
    # 작은 데이터로 테스트해보기


Epoch 1: loss=0.7673, acc=0.4922


KeyboardInterrupt: 

In [ ]:
# 4번 문항
pad_idx = vocab["<pad>"]
naive_transformer_model = TransformerClassifier(
    vocab_size=len(vocab),
    embedding_dim=300,
    num_heads=4,
    hidden_dim=256,
    num_layers=2,
    pad_idx=pad_idx,
).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(naive_transformer_model.parameters(), lr=1e-3)

In [ ]:
# 5번 문항
pad_idx = vocab["<pad>"]
naive_transformer_model = TransformerClassifier(
    vocab_size=len(vocab),
    embedding_dim=300,
    num_heads=4,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3,
    pad_idx=pad_idx,
).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(naive_transformer_model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
naive_transformer_model = train_loop(
    model=naive_transformer_model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10,
)


Epoch 1/10


Training: 100%|██████████| 391/391 [02:54<00:00,  2.25it/s, loss=0.446]


Epoch 1 Summary - Avg Loss: 0.5699, Train Acc: 0.6885, Test Acc: 0.7305

Epoch 2/10


Training: 100%|██████████| 391/391 [02:53<00:00,  2.25it/s, loss=0.58]


Epoch 2 Summary - Avg Loss: 0.4564, Train Acc: 0.7852, Test Acc: 0.7870

Epoch 3/10


Training: 100%|██████████| 391/391 [02:53<00:00,  2.25it/s, loss=0.31]


... (11 outputs left out)

Training: 100%|██████████| 391/391 [02:56<00:00,  2.21it/s, loss=0.258]


Epoch 9 Summary - Avg Loss: 0.2099, Train Acc: 0.9165, Test Acc: 0.8414

Epoch 10/10


Training: 100%|██████████| 391/391 [02:56<00:00,  2.21it/s, loss=0.144]


Epoch 10 Summary - Avg Loss: 0.1877, Train Acc: 0.9249, Test Acc: 0.8444


In [ ]:
!pip uninstall -y transformers torch_xla sentence-transformers
!pip install -q --no-cache-dir --force-reinstall "transformers==4.41.0"

Found existing installation: transformers 4.41.0
Uninstalling transformers-4.41.0:
  Successfully uninstalled transformers-4.41.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 284.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 265.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 292.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 410.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 281.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 301.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 417.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 413.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 378.4 MB

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print("✔ Transformers OK")
# check 한번 해주기

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✔ Transformers OK


In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    AutoConfig,
)
from torch import optim
from tqdm.auto import tqdm

MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
test_texts  = test_df["text"].tolist()
test_labels = test_df["label"].tolist()
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=320):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            max_length=self.max_len,
            return_tensors="pt",
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        enc["labels"] = torch.tensor(label, dtype=torch.long)
        return enc
BATCH_SIZE = 16
train_dataset = IMDBDataset(train_texts, train_labels, tokenizer)
test_dataset  = IMDBDataset(test_texts,  test_labels,  tokenizer)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

device = cuda


In [ ]:
DROPOUT = 0.3

config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = 2
config.hidden_dropout_prob = DROPOUT
config.attention_probs_dropout_prob = DROPOUT

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
).to(device)

EPOCHS = 4
LR = 2e-5
WEIGHT_DECAY = 0.01  # L2 정규화

optimizer = optim.AdamW(
    bert_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = EPOCHS * len(train_loader)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def evaluate(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

            preds = torch.argmax(logits, dim=-1)
            labels = batch["labels"]

            correct += (preds == labels).sum().item()
            total += labels.size(0)
            total_loss += loss.item()

    return total_loss / len(data_loader), correct / total


# 6. 학습 루프
for epoch in range(1, EPOCHS + 1):
    bert_model.train()
    total_train_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = bert_model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        total_train_loss += loss.item()
        progress_bar.set_postfix(loss=float(loss.item()))

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_acc = evaluate(bert_model, test_loader, device)

    print(
        f"\nEpoch {epoch} Summary - "
        f"Train Loss: {avg_train_loss:.4f}, "
        f"Test Loss: {val_loss:.4f}, "
        f"Test Acc: {val_acc:.4f}"
    )

Epoch 1/4:   0%|          | 0/1563 [00:00<?, ?it/s]


Epoch 1 Summary - Train Loss: 0.3475, Test Loss: 0.1832, Test Acc: 0.9291


Epoch 2/4:   0%|          | 0/1563 [00:00<?, ?it/s]


Epoch 2 Summary - Train Loss: 0.2033, Test Loss: 0.1713, Test Acc: 0.9368


Epoch 3/4:   0%|          | 0/1563 [00:00<?, ?it/s]


Epoch 3 Summary - Train Loss: 0.1693, Test Loss: 0.1743, Test Acc: 0.9423


Epoch 4/4:   0%|          | 0/1563 [00:00<?, ?it/s]


Epoch 4 Summary - Train Loss: 0.1397, Test Loss: 0.1874, Test Acc: 0.9421


In [ ]:
SAVE_DIR = "./my_model"

bert_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

('./my_model/tokenizer_config.json',
 './my_model/special_tokens_map.json',
 './my_model/vocab.json',
 './my_model/merges.txt',
 './my_model/added_tokens.json',
 './my_model/tokenizer.json')

In [ ]:
# %%writefile test.py
# import sys
# import os
# import torch
# from torch.utils.data import Dataset, DataLoader
# from transformers import (
#     AutoTokenizer,
#     AutoModelForSequenceClassification,
#     DataCollatorWithPadding,
# )

# # ---------------- 1. 인자 읽기 ----------------
# if len(sys.argv) != 3:
#     print("Usage: python test.py [data_dir] [model_name]")
#     sys.exit(1)

# data_dir = sys.argv[1]
# model_name = sys.argv[2]

# # ---------------- 2. 모델 로드 ----------------
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("device =", device)
# print("model_name =", model_name)

# if model_name == "my_model":
#     model_path = "./my_model"
#     tokenizer = AutoTokenizer.from_pretrained(model_path)
#     model = AutoModelForSequenceClassification.from_pretrained(model_path)
# else:
#     raise ValueError(f"Unknown model_name: {model_name}")

# model.to(device)
# model.eval()

# # ---------------- 3. test 데이터셋 ----------------
# class IMDBTestDataset(Dataset):
#     def __init__(self, texts, tokenizer, max_len=320):
#         self.texts = texts
#         self.tokenizer = tokenizer
#         self.max_len = max_len

#     def __len__(self):
#         return len(self.texts)

#     def __getitem__(self, idx):
#         text = self.texts[idx]
#         enc = self.tokenizer(
#             text,
#             truncation=True,
#             padding=False,
#             max_length=self.max_len,
#             return_tensors="pt",
#         )
#         enc = {k: v.squeeze(0) for k, v in enc.items()}
#         return enc

# test_root = os.path.join(data_dir, "test")

# test_texts = []
# file_paths = []

# for subdir, _, files in os.walk(test_root):
#     for fname in files:
#         if not fname.endswith(".txt"):
#             continue
#         full_path = os.path.join(subdir, fname)
#         file_paths.append(full_path)

# file_paths = sorted(file_paths)

# for path in file_paths:
#     with open(path, "r", encoding="utf-8") as f:
#         text = f.read().strip()
#         test_texts.append(text)

# print(f"#test samples = {len(test_texts)}")

# test_dataset = IMDBTestDataset(test_texts, tokenizer)
# data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=16,
#     shuffle=False,
#     collate_fn=data_collator,
# )

# # ---------------- 4. inference + result.tsv ----------------
# results = []
# idx = 0

# with torch.no_grad():
#     for batch in test_loader:
#         batch = {k: v.to(device) for k, v in batch.items()}
#         outputs = model(**batch)
#         logits = outputs.logits
#         preds = torch.argmax(logits, dim=-1)

#         for p in preds.cpu().tolist():
#             label_str = "pos" if p == 1 else "neg"
#             results.append((idx, label_str))
#             idx += 1

# with open("result.tsv", "w", encoding="utf-8") as f:
#     for i, label in results:
#         f.write(f"{i}\t{label}\n")

# print("Saved predictions to result.tsv")
#  지속적으로 NULL 오류가 발생하여 test.py를 완전히 재입력하는 코드를 혹시 몰라 추가해놨습니다.

In [31]:
%cd "/content/drive/MyDrive/Colab Notebooks"
!python test.py "/content/aclImdb" "my_model"

/content/drive/MyDrive/Colab Notebooks

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/content/drive/MyDrive/Colab Notebooks/test.py", line 3, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 1477, in <module>
    from .functional import *  # noqa: F403
  File "/usr/local/lib/python3.12/dist-packages/torch/functional.py", line 9, in <module>
    import torch.nn.functional as F
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/__init__.py", line 1, in <module>
    from .modules import *  # noqa: F403
  